# 03 - Exploratory Data Analysis & Feature Engineering

## CASEFILE: AI-Powered Missing Person Investigation System

### Investigative Objectives
In missing person investigations, establishing a subject's **Pattern of Life** is critical to locating them quickly. When someone vanishes, law enforcement must determine:
- Where do they spend their time (anchor points like home, workplace, frequent social spots)?
- What kinematics (speed, acceleration, heading changes) characterize their routine travel?
- What are their established temporal habits by hour of day and day of week?
- Where along their daily routes did their trajectory diverge from the baseline?

This notebook documents the **Feature Engineering and Exploratory Data Analysis (EDA) Phase**, analyzing engineered kinematic features, segmented trajectories, stay points, and behavioral user profiles.

### Feature Engineering Architecture (`src/feature_engineering.py`)
The feature engineering module converts cleaned GPS coordinates into semantically rich mobility indicators:
1. **Point-Level Kinematics**:
   - `distance_km`: Great-circle step displacement between consecutive pings.
   - `time_delta_seconds`: Sampling duration between fixes.
   - `speed_kmh`: Instantaneous velocity ($\text{distance} / \text{time}$).
   - `acceleration`: Linear acceleration rate ($\Delta v / \Delta t$).
   - `bearing`: Compass direction of heading in degrees ($0^\circ - 360^\circ$).
   - `bearing_change`: Absolute angular deflection ($0^\circ - 180^\circ$).
2. **Trajectory Segmentation**:
   - Splits continuous ping sequences into discrete journeys when temporal gaps exceed 20 minutes ($> 1200\text{s}$).
   - Computes journey duration, total distance, average/max speed, and `sinuosity` (path tortuosity: $\text{Actual Distance} / \text{Displacement}$).
3. **Stay-Point Detection Algorithm**:
   - Groups consecutive fixes where an individual remains within a spatial threshold ($< 100\text{m}$) for at least 5 minutes ($\ge 300\text{s}$).
4. **User Behavioral Profiling**:
   - Identifies home anchors (predominant night stay cluster 22:00–06:00), work anchors (weekday daytime cluster 09:00–17:00), and top active hours.
5. **Spatial Binning & Area IDs**:
   - Discretizes spatial locations into uniform grid cells ($0.005^\circ \approx 500\text{m}$) mapped to discrete area IDs.

In [ ]:
import os
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set aesthetic styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

print("Environment initialized and sys.path configured.")

### 1. Loading Processed Feature Datasets
We load the 4 core feature datasets from `data/processed/`:
1. `gps_features.csv`: Full point-level kinematic and grid features (**896,819 rows**)
2. `stay_points.csv`: Extracted spatial dwell events (**2,102 rows**)
3. `user_profiles.csv`: User mobility summaries and resolved home/work anchors (**10 profiles**)
4. `trajectory_summary.csv`: Segmented journey metrics (**32 trajectories**)

In [ ]:
import sys
sys.path.insert(0, '..')

proc_dir = os.path.join('..', 'data', 'processed')

df_features = pd.read_csv(os.path.join(proc_dir, 'gps_features.csv'))
df_stays = pd.read_csv(os.path.join(proc_dir, 'stay_points.csv'))
df_profiles = pd.read_csv(os.path.join(proc_dir, 'user_profiles.csv'))
df_trajs = pd.read_csv(os.path.join(proc_dir, 'trajectory_summary.csv'))

print(f"• gps_features.csv:        {len(df_features):,} records, {df_features.shape[1]} columns")
print(f"• stay_points.csv:         {len(df_stays):,} detected stay points")
print(f"• user_profiles.csv:       {len(df_profiles):,} user profiles")
print(f"• trajectory_summary.csv:  {len(df_trajs):,} segmented journeys")

print("\n--- Sample Point-Level Kinematic Features ---")
display(df_features[['latitude', 'longitude', 'distance_km', 'speed_kmh', 'acceleration', 'bearing', 'area_id']].head())

### 2. Kinematic Feature Distributions (Speed, Distance, Bearing)
Kinematic features distinguish stationary dwell, pedestrian transit, and vehicular commutes:
- **Speed (km/h)**: Identifies transport modalities (walking $< 5\text{ km/h}$, vehicular $20 - 60\text{ km/h}$).
- **Step Distance (km)**: Reflects fix sampling regularity.
- **Bearing (Degrees)**: Evaluates directional orientation and corridor alignment.

In [ ]:
import sys
sys.path.insert(0, '..')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Speed distribution for moving points
moving_speeds = df_features[df_features['speed_kmh'] > 0.5]['speed_kmh']
sns.histplot(moving_speeds[moving_speeds <= 70], bins=35, kde=True, ax=axes[0], color='#1f77b4')
axes[0].set_title('Transit Speed Distribution (> 0.5 km/h)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Speed (km/h)', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].axvline(moving_speeds.median(), color='red', linestyle='--', label=f'Median: {moving_speeds.median():.1f} km/h')
axes[0].legend()

# 2. Step distance distribution
distances = df_features['distance_km'].dropna()
sns.histplot(distances[distances <= 0.8], bins=35, kde=True, ax=axes[1], color='#2ca02c')
axes[1].set_title('Step Distance Distribution (≤ 0.8 km)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Distance (km)', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].axvline(distances.median(), color='red', linestyle='--', label=f'Median: {distances.median():.3f} km')
axes[1].legend()

# 3. Directional bearing distribution
bearings = df_features['bearing'].dropna()
sns.histplot(bearings, bins=36, kde=False, ax=axes[2], color='#ff7f0e')
axes[2].set_title('Compass Bearing Distribution (0° - 360°)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Bearing (Degrees from North)', fontsize=11)
axes[2].set_ylabel('Count', fontsize=11)

plt.tight_layout()
plt.show()

### 3. Stay Point Analysis & Dwell Clusters
Stay points indicate sustained geographic presence (e.g. residences, office complexes, commercial dining centers). Analyzing dwell durations reveals anchor locations critical for missing person search prioritizing.

In [ ]:
import sys
sys.path.insert(0, '..')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Dwell time distribution (in hours)
dwell_hours = df_stays['dwell_time_minutes'] / 60.0
sns.histplot(dwell_hours[dwell_hours <= 12], bins=30, kde=True, ax=axes[0], color='#805ad5')
axes[0].set_title('Stay Point Dwell Time Distribution (≤ 12 Hours)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Dwell Time (Hours)', fontsize=11)
axes[0].set_ylabel('Number of Stay Events', fontsize=11)
axes[0].axvline(dwell_hours.median(), color='darkred', linestyle='--', 
                label=f'Median Dwell: {dwell_hours.median():.2f} hrs')
axes[0].legend()

# 2. Geographic stay points scatter plot across Beijing
scatter = axes[1].scatter(
    df_stays['stay_lon'], df_stays['stay_lat'],
    c=df_stays['user_id'], cmap='tab10',
    s=np.clip(df_stays['dwell_time_minutes'] / 15, 10, 200),
    alpha=0.65, edgecolors='black', linewidths=0.5
)
axes[1].set_title('Stay Points Across Beijing (Size ~ Dwell Duration)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Longitude (°E)', fontsize=11)
axes[1].set_ylabel('Latitude (°N)', fontsize=11)
cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label('User ID', fontsize=10)
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### 4. Segmented Trajectory Statistics
From `trajectory_summary.csv`, we analyze journey-level metrics:
- Total journey distance vs. duration
- **Sinuosity** (ratio of actual traversed distance to straight-line displacement: $S = \frac{D_{\text{actual}}}{D_{\text{euclidean}}}$). Values near 1.0 indicate direct commutes along arterials; high values signify localized wandering or circuitous activity.

In [ ]:
import sys
sys.path.insert(0, '..')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Journey duration vs distance
sns.scatterplot(
    data=df_trajs, x='duration_minutes', y='total_distance_km',
    hue='user_id', palette='tab10', s=70, ax=axes[0]
)
axes[0].set_title('Journey Duration vs Total Distance', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Duration (Minutes)', fontsize=11)
axes[0].set_ylabel('Total Distance (km)', fontsize=11)
axes[0].grid(True, linestyle='--', alpha=0.5)

# 2. Sinuosity distribution on log scale
log_sinuosity = np.log10(df_trajs['sinuosity'].clip(lower=1.0))
sns.histplot(log_sinuosity, bins=25, kde=True, ax=axes[1], color='#319795')
axes[1].set_title('Trajectory Sinuosity Distribution (log10 scale)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log10(Sinuosity = Path Length / Displacement)', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)

plt.tight_layout()
plt.show()

### 5. User Behavioral Profiles Summary Table
The user profiling module (`compute_user_profiles`) aggregates historical behavior into an operational intelligence summary for each subject.

In [ ]:
import sys
sys.path.insert(0, '..')

display_cols = [
    'user_id', 'total_trajectories', 'total_distance_km', 'avg_daily_distance_km',
    'avg_speed_kmh', 'num_unique_stay_points', 'active_hours', 
    'home_lat', 'home_lon', 'work_lat', 'work_lon'
]

profiles_formatted = df_profiles[display_cols].copy()
profiles_formatted['total_distance_km'] = profiles_formatted['total_distance_km'].round(1)
profiles_formatted['avg_daily_distance_km'] = profiles_formatted['avg_daily_distance_km'].round(1)
profiles_formatted['avg_speed_kmh'] = profiles_formatted['avg_speed_kmh'].round(1)
profiles_formatted['home_lat'] = profiles_formatted['home_lat'].round(4)
profiles_formatted['home_lon'] = profiles_formatted['home_lon'].round(4)
profiles_formatted['work_lat'] = profiles_formatted['work_lat'].round(4)
profiles_formatted['work_lon'] = profiles_formatted['work_lon'].round(4)

print("Consolidated User Profiles Summary Table:")
display(profiles_formatted)

### 6. Movement Patterns by Hour of Day and Day of Week
Establishing temporal mobility patterns lets investigators rapidly evaluate whether an observed sighting or last-known ping occurred during a normal routine or represents an abnormal excursion.

In [ ]:
import sys
sys.path.insert(0, '..')

# Activity matrix: Day of Week vs Hour of Day
dow_labels = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
activity_pivot = df_features.groupby(['day_of_week', 'hour']).size().unstack(fill_value=0)
activity_pivot.index = [dow_labels[i] for i in activity_pivot.index]

plt.figure(figsize=(15, 6))
sns.heatmap(activity_pivot, cmap='YlGnBu', cbar_kws={'label': 'Observed GPS Pings'})
plt.title('Baseline Temporal Mobility Rhythm: GPS Activity Heatmap (Day of Week vs Hour)', fontsize=14, fontweight='bold')
plt.xlabel('Hour of Day (00:00 - 23:00)', fontsize=12)
plt.ylabel('Day of Week', fontsize=12)
plt.tight_layout()
plt.show()

### Summary & Downstream Investigation Value
- **Kinematics**: Established velocity baselines (median commute speed $\approx 3.5\text{ km/h}$ across stops and transit) and directional bearing trends.
- **Habitual Anchors**: Extracted **2,102** stay point occurrences, pinpointing primary residences, work campuses, and dining hubs for all 10 subjects.
- **Journey Metrics**: Segmented trajectories into **32** distinct multi-hour trips with sinuosity profiling to distinguish regular routes from wandering.
- **Investigation Readiness**: These engineered features directly feed:
  - **DBSCAN Clustering** (`src/clustering.py`): Discovering recurring geographic hubs.
  - **Anomaly Detection** (`src/anomaly_detection.py`): Pinpointing unusual deviations.
  - **Route & Area Prediction** (`src/route_prediction.py`, `src/prediction.py`): Prioritizing high-probability search sectors in active missing person cases.